In [ ]:
# CAMB/EFTCAMB constants and density helpers (Python version of modules.f90 logic)
import math
from typing import Dict, List, Sequence

# Fixed constants (EFTCAMB/constants.f90)
c = 2.99792458e8               # speed of light [m/s]
G = 6.6738e-11                 # Newton's constant [SI]
kappa = 8.0 * math.pi * G      # 8*pi*G
sigma_boltz = 5.6704e-8        # Stefan-Boltzmann constant
Mpc = 3.085678e22              # 1 Mpc in meters
COBE_CMBTemp = 2.7255          # default T_CMB (K)
default_nnu = 3.046            # default N_eff

def camb_today_densities(
    hubble: float,
    omegab: float,
    omegac: float,
    omegav: float,
    omegak: float,
    tcmb: float = COBE_CMBTemp,
    num_nu_massless: float = default_nnu,
    num_nu_massive: int = 0,
    nu_mass_numbers: Sequence[int] | None = None,
    share_delta_neff: bool = True,
) -> Dict[str, object]:
    """Replicates EFTCAMB/modules.f90 density setup at a=1.

    Returns grhob/grhoc/grhog/grhornomass/grhormass/grhov/grhok/grhom/grhor.
    """
    h0_Mpc = hubble * 1000.0 / c
    grhom = 3.0 * h0_Mpc**2

    grhog = kappa / c**2 * 4.0 * sigma_boltz / c**3 * (tcmb**4) * Mpc**2
    grhor = (7.0 / 8.0) * (4.0 / 11.0) ** (4.0 / 3.0) * grhog

    if nu_mass_numbers is None:
        nu_mass_numbers = [0] * num_nu_massive

    nu_massless_degeneracy = num_nu_massless
    nu_mass_degeneracies: List[float] = list(nu_mass_numbers)
    if num_nu_massive > 0 and share_delta_neff:
        fractional_number = num_nu_massless + num_nu_massive
        actual_massless = int(num_nu_massless + 1e-6)
        neff_i = fractional_number / (actual_massless + num_nu_massive)
        nu_massless_degeneracy = neff_i * actual_massless
        nu_mass_degeneracies = [n * neff_i for n in nu_mass_numbers]

    grhornomass = grhor * nu_massless_degeneracy
    grhormass = [grhor * g for g in nu_mass_degeneracies]

    grhob = grhom * omegab
    grhoc = grhom * omegac
    grhov = grhom * omegav
    grhok = grhom * omegak

    return {
        "grhom": grhom,
        "grhob": grhob,
        "grhoc": grhoc,
        "grhov": grhov,
        "grhok": grhok,
        "grhog": grhog,
        "grhor": grhor,
        "grhornomass": grhornomass,
        "grhormass": grhormass,
    }

def camb_scale_factor_scaling(
    a: float,
    grhob: float,
    grhoc: float,
    grhog: float,
    grhornomass: float,
    grhormass_list: Sequence[float] | None = None,
):
    """Scale densities to time a (rho_i * a^2 / m0^2, as used in EFTCAMB)."""
    if grhormass_list is None:
        grhormass_list = []
    a2 = a * a
    grhob_t = grhob / a
    grhoc_t = grhoc / a
    grhor_t = grhornomass / a2
    grhog_t = grhog / a2
    grhormass_t = [g / a2 for g in grhormass_list]
    return grhob_t, grhoc_t, grhog_t, grhor_t, grhormass_t


In [3]:
# Pade series helper (pure Python, no ini): supply coefficients directly
from dataclasses import dataclass
from typing import List


def poly_eval(coeff: List[float], x: float) -> float:
    """Evaluate polynomial with ascending-order coefficients using Horner."""
    res = 0.0
    for c in reversed(coeff):
        res = res * x + c
    return res


def derivative_coeffs(coeff: List[float], order: int) -> List[float]:
    """Return coefficients of the requested derivative (ascending order)."""
    cur = list(coeff)
    for _ in range(order):
        cur = [i * c for i, c in enumerate(cur)][1:]
        if not cur:
            return [0.0]
    return cur


@dataclass
class PadeFunction:
    name: str
    a0: float
    coeff_up: List[float]      # ascending order in (a - a0)
    coeff_down: List[float]    # first element is constant term
    order_up: int | None = None
    order_down: int | None = None

    def __post_init__(self):
        # Infer orders if not provided
        self.order_up = len(self.coeff_up) - 1 if self.order_up is None else self.order_up
        self.order_down = len(self.coeff_down) - 1 if self.order_down is None else self.order_down
        # Ensure denominator constant term is 1 like EFTCAMB; rescale if needed
        if self.coeff_down and self.coeff_down[0] == 0:
            raise ValueError('Denominator constant term must be non-zero (typically 1).')

    @classmethod
    def from_coeffs(cls, name: str, a0: float, coeff_up: List[float], coeff_down: List[float]):
        """Factory helper mirroring EFTCAMB structure."""
        return cls(name=name, a0=a0, coeff_up=list(coeff_up), coeff_down=list(coeff_down))

    def _shift(self, a: float) -> float:
        return a - self.a0

    def _taylor_up(self, a: float) -> float:
        return poly_eval(self.coeff_up, self._shift(a))

    def _taylor_down(self, a: float) -> float:
        return poly_eval(self.coeff_down, self._shift(a))

    def _taylor_up_der(self, a: float, order: int) -> float:
        return poly_eval(derivative_coeffs(self.coeff_up, order), self._shift(a))

    def _taylor_down_der(self, a: float, order: int) -> float:
        return poly_eval(derivative_coeffs(self.coeff_down, order), self._shift(a))

    def value(self, a: float) -> float:
        return self._taylor_up(a) / self._taylor_down(a)

    def first_derivative(self, a: float) -> float:
        up = self._taylor_up(a)
        dup = self._taylor_up_der(a, 1)
        down = self._taylor_down(a)
        ddown = self._taylor_down_der(a, 1)
        return (dup * down - up * ddown) / (down ** 2)

    def second_derivative(self, a: float) -> float:
        up = self._taylor_up(a)
        dup = self._taylor_up_der(a, 1)
        ddup = self._taylor_up_der(a, 2)
        down = self._taylor_down(a)
        ddown = self._taylor_down_der(a, 1)
        dddown = self._taylor_down_der(a, 2)
        return (ddup * down ** 2 - 2 * ddown * down * dup + 2 * ddown ** 2 * up - dddown * down * up) / (down ** 3)

    def third_derivative(self, a: float) -> float:
        up = self._taylor_up(a)
        dup = self._taylor_up_der(a, 1)
        ddup = self._taylor_up_der(a, 2)
        dddup = self._taylor_up_der(a, 3)
        down = self._taylor_down(a)
        ddown = self._taylor_down_der(a, 1)
        dddown = self._taylor_down_der(a, 2)
        ddddown = self._taylor_down_der(a, 3)
        return (-3 * ddown * ddup * down ** 2 + dddup * down ** 3 + 6 * ddown ** 2 * down * dup
                - 3 * dddown * down ** 2 * dup - 6 * ddown ** 3 * up + 6 * dddown * ddown * down * up
                - ddddown * down ** 2 * up) / (down ** 4)

    def integral(self, a: float) -> float:
        raise NotImplementedError('Integral not implemented (same as EFTCAMB).')


# Example: P(a) = (N0 + N1 (a-a0)) / (1 + D1 (a-a0))
pade = PadeFunction.from_coeffs('Omega', a0=1.0, coeff_up=[0.1, 0.2, 0.3], coeff_down=[1.0, -0.3])
pade.first_derivative(0)
# pade.value(0)


-0.27218934911242604

In [ ]:
# ODE solver for y(a) using SciPy LSODA (A d/dln a y + B y + C = 0)
import numpy as np
from scipy.integrate import solve_ivp

# physical constants (SI)
C_LIGHT = 299792458.0
G_NEWTON = 6.67430e-11
MPC_IN_METERS = 3.085677581491367e22
# 1/m0^2 = 8 pi G / c^2 (consistent with EFTCAMB usage of kappa/c^2)
KAPPA_OVER_C2 = 8.0 * np.pi * G_NEWTON / (C_LIGHT ** 2)


def solve_y_lsoda(
    omega,            # Ω(a)
    omega_p,          # Ω'(a)
    omega_pp,         # Ω''(a)
    Lambda,           # Λ(a)
    pm_term,          # callable P_m a^2 / m0^2 (already scaled, no extra a/m0 factors)
    y0: float,        # initial y at a_start (y = H^2)
    a_span: tuple,    # (a_start, a_end)
    atol: float = 1e-9,
    rtol: float = 1e-7,
    max_step=None,
    kappa_over_c2: float = KAPPA_OVER_C2,
    lambda_unit_factor: float = MPC_IN_METERS ** 2,
):
    """Solve A d/d ln a y + B y + (Pm_term + Lambda_term) = 0 with LSODA.

    - A = 1 + Ω + 0.5 a Ω'
    - B = 1 + Ω + 2 a Ω' + a^2 Ω''
    - Lambda_term = (kappa/c^2) * Λ(a) * a^2 * lambda_unit_factor
      (matching EFTCAMB: Lambda tabulated in 1/Mpc^2 -> multiply by Mpc^2)
    - pm_term(a) should already be P_m a^2 / m0^2 (do NOT multiply by a or m0 here).
    """

    a0, a1 = a_span
    max_step_val = np.inf if max_step is None else max_step

    def rhs(a, y):
        Om = omega(a)
        Op = omega_p(a)
        Opp = omega_pp(a)
        lam = Lambda(a)

        Acoef = 1.0 + Om + 0.5 * a * Op
        Bcoef = 1.0 + Om + 2.0 * a * Op + a * a * Opp
        Cterm = pm_term(a) + kappa_over_c2 * lam * a * a * lambda_unit_factor

        dy_dloga = -(Bcoef * y[0] + Cterm) / Acoef
        return [dy_dloga / a]  # convert d/d ln a -> d/da

    sol = solve_ivp(
        rhs,
        t_span=(a0, a1),
        y0=[y0],
        method='LSODA',
        atol=atol,
        rtol=rtol,
        max_step=max_step_val,
        dense_output=True,
    )
    return sol


# Example usage (replace with real functions):
# omega    = lambda a: 0.1 * a
# omega_p  = lambda a: 0.1
# omega_pp = lambda a: 0.0
# Lambda   = lambda a: 0.0
# pm_term  = lambda a: 0.01   # already P_m a^2 / m0^2
# res = solve_y_lsoda(omega, omega_p, omega_pp, Lambda, pm_term, y0=1.0, a_span=(1.0, 2.0))
# a_eval = np.linspace(1.0, 2.0, 50)
# y_eval = res.sol(a_eval)[0]


In [ ]:
# Background output helper mirroring EFTCAMBHorndeskiSolveBackgroundEquations::output
import math

C_LIGHT = 299792458.0
G_NEWTON = 6.67430e-11
MPC_IN_METERS = 3.085677581491367e22
KAPPA = 8.0 * math.pi * G_NEWTON / (C_LIGHT ** 4)
KAPPA_OVER_C2 = KAPPA * (C_LIGHT ** 2)  # = 8piG / c^2


def horndeski_output(
    a: float,
    H2: float,
    ydot: float,                # dH^2/d ln a
    Omega: float,
    Omegap: float,
    Omegapp: float,
    Lambda: float,
    Lambda_p: float,
    grho_matter: float,         # already scaled like EFT: ρ_m a^2 / m0^2
    gpres_matter: float,        # same scaling for pressure term P_m a^2 / m0^2
    kappa_over_c2: float = KAPPA_OVER_C2,
    lambda_unit_factor: float = MPC_IN_METERS ** 2,
):
    """Port of EFTCAMBHorndeskiSolveBackgroundEquations output() to Python.

    All inputs follow EFTCAMB internal scaling: grho_matter = ρ_m a^2 / m0^2, etc.
    Lambda is the EFT function (units 1/Mpc^2 typically), converted with lambda_unit_factor.
    Returns dict with keys: ca2_over_m0sq, Lambda_a2, cdot_a2_over_m0sq, Lambda_a2_prime,
    rhoDE_hat, pDE_hat, OmegaDE, wDE, rhoDE_real, pDE_real, c_real, H_phys.
    """
    a2 = a * a

    # Lambda a^2 / m0^2
    Lambda_a2 = kappa_over_c2 * Lambda * a2 * lambda_unit_factor
    Lambda_a2_prime = kappa_over_c2 * lambda_unit_factor * (2.0 * a * Lambda + a2 * Lambda_p)

    # c(a) hat: ca^2/m0^2
    ca2_over_m0sq = 1.5 * (1.0 + Omega + a * Omegap) * H2                     - 0.5 * grho_matter                     + 0.5 * Lambda_a2

    # time derivatives (d/dη = 𝓗 d/dN) if H2 > 0
    if H2 > 0.0:
        a2Omegapp = a2 * Omegapp
        d_factor_dN = 2.0 * a * Omegap + a2Omegapp
        drho_ma2_dN = -(grho_matter + 3.0 * gpres_matter)
        dLambda_a2_dN = a * Lambda_a2_prime
        dC_dN = 1.5 * (1.0 + Omega + a * Omegap) * ydot                 + 1.5 * H2 * d_factor_dN                 - 0.5 * drho_ma2_dN                 + 0.5 * dLambda_a2_dN
        cdot_a2_over_m0sq = math.sqrt(H2) * dC_dN
        Lambdadot_a2_over_m0sq = math.sqrt(H2) * dLambda_a2_dN
    else:
        cdot_a2_over_m0sq = math.nan
        Lambdadot_a2_over_m0sq = math.nan

    # hat density and pressure
    rhoDE_hat = 3.0 * H2 - grho_matter
    Hdot = 0.5 * ydot
    pDE_hat = -2.0 * Hdot - H2 - gpres_matter

    # real quantities: m0^2/a^2 = c^2 / (kappa * Mpc^2) / a^2
    pref_m0_over_a2 = ((C_LIGHT ** 2) / (KAPPA * (MPC_IN_METERS ** 2))) / a2
    rhoDE_real = pref_m0_over_a2 * rhoDE_hat
    pDE_real = pref_m0_over_a2 * pDE_hat
    c_real = pref_m0_over_a2 * ca2_over_m0sq

    if H2 > 0.0:
        OmegaDE = rhoDE_hat / (3.0 * H2)
    else:
        OmegaDE = math.nan

    if H2 > 0.0 and abs(rhoDE_hat) > 1e-16 * 3.0 * H2:
        wDE = pDE_hat / rhoDE_hat
    else:
        wDE = math.nan

    H_phys = math.sqrt(H2) / a

    return {
        'ca2_over_m0sq': ca2_over_m0sq,
        'Lambda_a2': Lambda_a2,
        'cdot_a2_over_m0sq': cdot_a2_over_m0sq,
        'Lambdadot_a2_over_m0sq': Lambdadot_a2_over_m0sq,
        'Lambda_a2_prime': Lambda_a2_prime,
        'rhoDE_hat': rhoDE_hat,
        'pDE_hat': pDE_hat,
        'OmegaDE': OmegaDE,
        'wDE': wDE,
        'rhoDE_real': rhoDE_real,
        'pDE_real': pDE_real,
        'c_real': c_real,
        'H_phys': H_phys,
    }


# Example call (fill with real background values):
# out = horndeski_output(
#     a=1.0, H2=1.0, ydot=-0.1,
#     Omega=0.1, Omegap=0.05, Omegapp=0.0,
#     Lambda=0.0, Lambda_p=0.0,
#     grho_matter=0.3, gpres_matter=0.0,
# )


In [ ]:
# ODE solver for y(a) using SciPy LSODA (A d/dln a y + B y + C = 0)
import numpy as np
from scipy.integrate import solve_ivp

# physical constants (SI)
C_LIGHT = 299792458.0
G_NEWTON = 6.67430e-11
MPC_IN_METERS = 3.085677581491367e22
# 1/m0^2 = 8 pi G / c^2 (consistent with EFTCAMB usage of kappa/c^2)
KAPPA_OVER_C2 = 8.0 * np.pi * G_NEWTON / (C_LIGHT ** 2)


def solve_y_lsoda(
    omega,            # Ω(a)
    omega_p,          # Ω'(a)
    omega_pp,         # Ω''(a)
    Lambda,           # Λ(a)
    pm_term,          # callable P_m a^2 / m0^2 (already scaled, no extra a/m0 factors)
    y0: float,        # initial y at a_start (y = H^2)
    a_span: tuple,    # (a_start, a_end)
    atol: float = 1e-9,
    rtol: float = 1e-7,
    max_step=None,
    kappa_over_c2: float = KAPPA_OVER_C2,
    lambda_unit_factor: float = MPC_IN_METERS ** 2,
):
    """Solve A d/d ln a y + B y + (Pm_term + Lambda_term) = 0 with LSODA.

    - A = 1 + Ω + 0.5 a Ω'
    - B = 1 + Ω + 2 a Ω' + a^2 Ω''
    - Lambda_term = (kappa/c^2) * Λ(a) * a^2 * lambda_unit_factor
      (matching EFTCAMB: Lambda tabulated in 1/Mpc^2 -> multiply by Mpc^2)
    - pm_term(a) should already be P_m a^2 / m0^2 (do NOT multiply by a or m0 here).
    """

    a0, a1 = a_span
    max_step_val = np.inf if max_step is None else max_step

    def rhs(a, y):
        Om = omega(a)
        Op = omega_p(a)
        Opp = omega_pp(a)
        lam = Lambda(a)

        Acoef = 1.0 + Om + 0.5 * a * Op
        Bcoef = 1.0 + Om + 2.0 * a * Op + a * a * Opp
        Cterm = pm_term(a) + kappa_over_c2 * lam * a * a * lambda_unit_factor

        dy_dloga = -(Bcoef * y[0] + Cterm) / Acoef
        return [dy_dloga / a]  # convert d/d ln a -> d/da

    sol = solve_ivp(
        rhs,
        t_span=(a0, a1),
        y0=[y0],
        method='LSODA',
        atol=atol,
        rtol=rtol,
        max_step=max_step_val,
        dense_output=True,
    )
    return sol


# Example usage (replace with real functions):
# omega    = lambda a: 0.1 * a
# omega_p  = lambda a: 0.1
# omega_pp = lambda a: 0.0
# Lambda   = lambda a: 0.0
# pm_term  = lambda a: 0.01   # already P_m a^2 / m0^2
# res = solve_y_lsoda(omega, omega_p, omega_pp, Lambda, pm_term, y0=1.0, a_span=(1.0, 2.0))
# a_eval = np.linspace(1.0, 2.0, 50)
# y_eval = res.sol(a_eval)[0]
